In [1]:
#Importing Libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

In [6]:
#Defining Image Parameters and Data Augmentation
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

data_dir = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog"  # Change to your dataset path
train_dir = data_dir + "/train"
val_dir = data_dir + "/validation"

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 800 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


In [7]:
#Loading VGG16 Model and Add Custom Layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
for layer in base_model.layers:
    layer.trainable = False

x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=x)


In [9]:
#Compiling and Train Model
model.compile(optimizer=SGD(learning_rate=0.001, momentum=0.9),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator
)

model.save("dog_cat_classifier.h5")

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 243s 10s/step - accuracy: 0.7454 - loss: 0.4865 - val_accuracy: 0.8300 - val_loss: 0.3771
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 242s 10s/step - accuracy: 0.8073 - loss: 0.4358 - val_accuracy: 0.8600 - val_loss: 0.3229
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 262s 10s/step - accuracy: 0.8377 - loss: 0.3917 - val_accuracy: 0.8950 - val_loss: 0.3056
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 275s 11s/step - accuracy: 0.8143 - loss: 0.4095 - val_accuracy: 0.8650 - val_loss: 0.3118
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 244s 10s/step - accuracy: 0.8057 - loss: 0.4004 - val_accuracy: 0.8650 - val_loss: 0.3101
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 285s 12s/step - accuracy: 0.7909 - loss: 0.3975 - val_accuracy: 0.8450 - val_loss: 0.3494
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 256s 10s/step - accuracy: 0.7994 - loss: 0.4334 - val_accuracy: 0.8850 - val_loss: 0.2898
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 257s 10s/step - accuracy: 0.8278 - loss: 0.3871 - val_accuracy: 0.

In [13]:
# Predicting the class for new Images
def predict_image(image_path, model):
    img = image.load_img(image_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize
    prediction = model.predict(img_array)
    class_name = "Dog" if prediction[0][0] > 0.5 else "Cat"
    confidence = prediction[0][0] if class_name == "Dog" else 1 - prediction[0][0]
    return class_name, confidence

img_path = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog/sample_1.jpg"  # Change to test image path
label, confidence = predict_image(img_path, model)
print(f"Predicted: {label} with confidence: {confidence:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
Predicted: Dog with confidence: 0.98


In [15]:
def predict_image(image_path, model):
    img = image.load_img(image_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize
    prediction = model.predict(img_array)
    class_name = "Dog" if prediction[0][0] > 0.5 else "Cat"
    confidence = prediction[0][0] if class_name == "Dog" else 1 - prediction[0][0]
    return class_name, confidence

img_path = "/content/drive/MyDrive/For Collab/Quiz_and_assignments/Cat_and_Dog/sample_2.jpg"  # Change to test image path
label, confidence = predict_image(img_path, model)
print(f"Predicted: {label} with confidence: {confidence:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step
Predicted: Dog with confidence: 1.00
